In [ ]:
import torch
from torch.nn import functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import tqdm
# Define a dummy dataset for relation extraction
class RelationExtractionDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=64):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        input_text, target_text = self.examples[idx]
        inputs = self.tokenizer(input_text, return_tensors="pt", padding='max_length', max_length=self.max_length, truncation=True)
        targets = self.tokenizer(target_text, return_tensors="pt", padding='max_length', max_length=self.max_length, truncation=True)
        return inputs.input_ids.squeeze(0), inputs.attention_mask.squeeze(0), targets.input_ids.squeeze(0)

# Define the EWC loss computation
def compute_ewc_loss(model, fisher, old_params, ewc_lambda):
    loss = 0.0
    for n, p in model.named_parameters():
        if n in fisher:
            loss += (fisher[n] * (p - old_params[n]) ** 2).sum()
    return ewc_lambda * loss

# Initialize tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Task A: Relation type "per:spouse"
task_a_data = [
    ("Extract the relation between Barack Obama and Michelle Obama", "per:spouse"),
    ("Extract the relation between Bill Gates and Melinda Gates", "per:spouse")
]

# Task B: Relation type "org:top_members/employees"
task_b_data = [
    ("Extract the relation between Sundar Pichai and Google", "org:top_members/employees"),
    ("Extract the relation between Tim Cook and Apple", "org:top_members/employees")
]

# Create datasets and dataloaders
train_dataset_a = RelationExtractionDataset(task_a_data, tokenizer)
train_loader_a = DataLoader(train_dataset_a, batch_size=2, shuffle=True)

train_dataset_b = RelationExtractionDataset(task_b_data, tokenizer)
train_loader_b = DataLoader(train_dataset_b, batch_size=2, shuffle=True)

# Optimizer
optimizer = AdamW(model.parameters(), lr=3e-5)

# Train on Task A
model.train()
for epoch in range(1):
    for input_ids, attention_mask, labels in train_loader_a:
        print(epoch)
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Save parameters and compute Fisher Information
old_params = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}
fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}

model.eval()
for input_ids, attention_mask, labels in train_loader_a:
    input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    for n, p in model.named_parameters():
        if p.grad is not None:
            fisher[n] += p.grad.detach() ** 2

for n in fisher:
    fisher[n] /= len(train_loader_a)
print("Task B")
# Train on Task B with EWC
model.train()
ewc_lambda = 1000

for epoch in range(10):
    print(epoch)
    for input_ids, attention_mask, labels in train_loader_b:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        task_loss = outputs.loss
        ewc_loss = compute_ewc_loss(model, fisher, old_params, ewc_lambda)
        total_loss = task_loss + ewc_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

In [ ]:
pip install torch transformers


In [ ]:
pip install datasets tqdm
